In [ ]:
# Step 1: Check conda environments
# Step 2: Switch to sagemaker-distribution
# Step 3: Pip install libraries 

In [ ]:
import requests
import pandas as pd
import io
import os

# Download Dataset

In [ ]:
def excel_to_csv(url, output_dir="."):
    """
    Downloads an Excel file from a given URL, converts it to CSV, and saves it.

    Args:
        url (str): The URL of the Excel file.
        output_dir (str, optional): The directory where the CSV file will be saved. Defaults to the current directory.

    Returns:
        str: The path to the saved CSV file, or None on error.
    """
    try:
        # 1. Download the Excel file
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes

        # Use io.BytesIO to handle the downloaded content as a file-like object
        excel_content = io.BytesIO(response.content)

        # 2. Read the Excel file into a pandas DataFrame
        #    -  Attempt to automatically detect all sheet names.
        xls = pd.read_excel(excel_content, sheet_name=None)

        # 3.  Convert each sheet to a CSV file.
        filepaths = []
        for sheet_name, df in xls.items():
            # Sanitize the sheet name to be used as a filename.
            safe_sheet_name = "".join(c if c.isalnum() else "_" for c in sheet_name)
            # Construct the output file path.
            csv_file_path = os.path.join(output_dir, f"{safe_sheet_name}.csv")

            # Save the DataFrame as a CSV file.
            df.to_csv(csv_file_path, index=False, encoding='utf-8')
            filepaths.append(csv_file_path)
        if not filepaths:
            print("No sheets found in the excel file")
            return None
        elif len(filepaths) == 1:
            return filepaths[0] # Return only the first file if there is only one sheet
        else:
            return filepaths # Return a list of filepaths


    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        return None
    except pd.errors.EmptyDataError:
        print("Error: The Excel file is empty.")
        return None
    except pd.errors.ParserError as e:
        print(f"Error parsing the Excel file: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def download_and_save_file(excel_url, outpur_directory):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    csv_file_path = excel_to_csv(excel_url, output_dir=output_directory)

    if csv_file_path:
        if isinstance(csv_file_path, list):
            print("Successfully converted the Excel file. CSV files created:")
            for path in csv_file_path:
                print(path)
        else:
            print(f"Successfully converted the Excel file to: {csv_file_path}")
    else:
        print("Failed to convert the Excel file.")

In [ ]:
# Dataset is derived from the Tableau Superstore dataset
excel_url = "https://www.tableau.com/sites/default/files/2021-05/Sample%20-%20Superstore.xls"
output_directory = "data"  # Specify the directory to save the CSV file

download_and_save_file(excel_url, output_directory)

# Set up your "Relational Database"

In [ ]:
def create_relational_dataframes(df):
    """
    Separates a Pandas DataFrame containing order, customer, and product information
    into three relational DataFrames: orders, customers, and products.

    Args:
        df: The input Pandas DataFrame.

    Returns:
        A dictionary containing the three relational DataFrames:
        {'orders': orders_df, 'customers': customers_df, 'products': products_df}
    """

    # --- Products DataFrame ---
    products_df = df[['Product ID', 'Category', 'Sub-Category', 'Product Name', 'Product Description']].drop_duplicates().copy()
    products_df.reset_index(drop=True, inplace=True)

    # --- Customers DataFrame ---
    customers_df = df[['Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State', 'Postal Code', 'Region']].drop_duplicates().copy()
    customers_df.reset_index(drop=True, inplace=True)

    # --- Orders DataFrame ---
    orders_df = df[['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Product ID', 'Sales', 'Quantity', 'Discount', 'Profit']].copy()
    orders_df.reset_index(drop=True, inplace=True)
    
    # Convert date columns to datetime objects
    orders_df['Order Date'] = pd.to_datetime(orders_df['Order Date'])
    orders_df['Ship Date'] = pd.to_datetime(orders_df['Ship Date'])

    return {'orders': orders_df, 'customers': customers_df, 'products': products_df}

In [ ]:
data_file = os.path.join("data", "Orders.csv")
superstore_df = pd.read_csv(data_file)

In [ ]:
superstore_df.head()

In [ ]:
# Create a new feature called "Product Description"
superstore_df['Product Description'] = 'This product is the category, ' +  superstore_df['Category'].astype(str) + ' and is specifically related to ' + superstore_df['Sub-Category'].astype(str) + '. The product is ' + superstore_df['Product Name'].astype(str)

In [ ]:
sql_database = create_relational_dataframes(superstore_df)

In [ ]:
sql_database['products'].head()

In [ ]:
orders = sql_database['orders']
customers = sql_database['customers']
products = sql_database['products']

In [ ]:
orders.head()

# Query your Database

In [ ]:
from pandasql import sqldf

In [ ]:
query = sqldf('''SELECT Category, "Product Name"
FROM products 
LIMIT 5''')

In [ ]:
query.head()

In [ ]:
query = sqldf('''SELECT DISTINCT `Order ID`,`Customer Name`, `Product Name`, `Quantity`
FROM orders
JOIN customers ON orders.`Customer ID` = customers.`Customer ID`
JOIN products on orders.`Product ID` = products.`Product ID`''')

In [ ]:
query.head()

In [ ]:
query = sqldf('''SELECT
    p."Product ID",
    p."Product Name",
    p.Category,
    p."Sub-Category",
    c.Segment,
    SUM(o.Sales) AS TotalSales
FROM
    orders o
JOIN
    customers c ON o."Customer ID" = c."Customer ID"
JOIN
    products p ON o."Product ID" = p."Product ID"
GROUP BY
    p."Product Name",
    p.Category,
    p."Sub-Category",
    c.Segment
ORDER BY
    TotalSales DESC;''')

In [ ]:
query.head(10)

# Download Your Sentence Embedding Model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Test that your embedding model is working
input_str = "I enjoy learning about data science and machine learning."
embeddings = embedding_model.embed_query(input_str)

In [ ]:
print(embeddings)

In [ ]:
print(len(embeddings))

# Select Data We Want to Vectorize

In [ ]:
from langchain.docstore.document import Document

In [ ]:
product_vector_df = sqldf('''SELECT DISTINCT "Product ID", "Product Description"
FROM products''')

In [ ]:
product_vector_df.head()

In [ ]:
documents = []
for index, row in product_vector_df.iterrows():
    content = row['Product Description']
    metadata = {"product_id": str(row['Product ID'])}  # Ensure product ID is a string
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

In [ ]:
documents[0]

# Set up local ChromaDB (vector database) for storing embeddings

In [ ]:
from langchain.vectorstores import Chroma

In [ ]:
persist_directory = os.path.join("data", "chroma_db")
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=persist_directory
)

# Filter based on data attributes and semantic search

In [ ]:
results = vectordb.similarity_search(
    "Office furniture well made and sturdy",
    k=5
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n")

In [ ]:
# Save results into a Pandas DataFrame
semantic_search_results = [
    {"content": res.page_content, **res.metadata} for res in results
]
semantic_search_results = pd.DataFrame(semantic_search_results)
semantic_search_results = semantic_search_results.rename(columns={
                                        "content": "Product Description",
                                        "product_id": "Product ID"})

# Display the DataFrame
print(semantic_search_results)

# Integrate with SQL results

In [ ]:
query = sqldf('''SELECT
    p."Product ID",
    p."Product Name",
    SUM(o.Sales) AS TotalSales
FROM
    orders o
JOIN
    products p ON o."Product ID" = p."Product ID"
GROUP BY
    p."Product Name"
ORDER BY
    TotalSales DESC;''')

In [ ]:
# Perform a left join
final_result = pd.merge(semantic_search_results, query, on='Product ID', how='left')

In [ ]:
final_result.sort_values(by="TotalSales", ascending=False)